In [2]:
# <eval_forecast_save.py>
"""
python /home/saptarishi.dhanuka_asp25/weather/graphcast_dir/graphcast/local_files/eval_forecast_save.py \ 
--eval_start "2014-08-01" \ 
--eval_end "2014-09-30" \ 
--eval_dataset_choice "imerg" \ 
--vars_to_eval "total_precipitation_6hr" \ 
--params_path_new1 "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2014-06-01_2014-07-30_FORECAST28_dynamic_weighing_india_mask_expt5.npz" \ 
--params_path_new2 "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2014-06-01_2014-07-30_FORECAST28_dynamic_weighing_india_mask_expt3.npz" \
--output_csv_path "./evaluation_results/forecast_mse.csv"
"""

"""
Complete evaluation of forecast against different datasets
"""
# <eval_forecast.py>
# Parameters
eval_start = "2014-08-01"
eval_end = "2014-09-05"
dataset_choice = "imerg"
eval_vars = "total_precipitation_6hr"
apath = "/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/"
params_path_old = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/origs/graphcast_1_13.npz'

params_path_new1 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_shapefile_2024-06-01_2024-07-30_FORECAST28_new.npz'
params_path_new2 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_shapefile_2024-06-01_2024-07-30_FORECAST28.npz'
params_paths = [params_path_new1, params_path_new2]
norms_dir = '/Datastorage/saptarishi.dhanuka_asp25/gc_norms/'
plots_dir = 'plots/evals'
latmin, latmax, lonmin, lonmax = 6, 38, 35, 65
plot_timesteps = 7
output_pred_old_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
output_pred_finetuned_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
region_vise = True
regions = ['Central_Northeast', 'Hilly_Regions', 'Northeast', 'Northwest', 'South_Peninsular', 'West_Central']
world_regions = ['India']
output_csv_path = './plots/skill_precip_data.csv'

"""
Complete evaluation of forecast against different datasets with rainfall analysis
"""
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import sys
import logging
import argparse
import dataclasses
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import time
import zarr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap
import glob

import jax
import optax


# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.80'
# os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# import utils
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
from graphcast import checkpoint, data_utils, rollout, graphcast, normalization
import setup_jax_functions
from plotting import scale, select, plot_data, save_animation, save_static_plot, compute_difference_with_targets_sims, plot_sample_from_ds
from metrics import compute_rmse, compute_mae, compute_bias, compute_acc
from utils import regrid_hres_fine_to_coarse, generate_sample_era5_dataset, grads_fn, parse_args, process_to_graphcast_format, compute_mse, compute_mse_diffs, mask_dbase_india_buffer, mask_dbase_regions

sys.path.append('/home/saptarishi.dhanuka_asp25/weather/graphcast_dir/gc_dist')
import trainer.dataloader
from dist_utils import construct_era5_imerg
from datetime import datetime

print("Imports done")




Imports done


In [3]:
files = sorted(glob.glob('/Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/target_init_2024-*'))
files[0],files[-1],len(files)

('/Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/target_init_2024-06-01 00:00:00.nc',
 '/Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/target_init_2024-09-23 00:00:00.nc',
 100)

In [4]:
combined = xr.open_dataset(files[0])
combined

/tmp/ipykernel_3573716/3131276784.py:1: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  combined = xr.open_dataset(files[0])


<xarray.Dataset> Size: 606MB
Dimensions:                  (batch: 1, time: 28, lat: 181, lon: 360, level: 13)
Coordinates:
    expver                   (time) <U4 448B ...
  * lat                      (lat) float64 1kB -90.0 -89.0 -88.0 ... 89.0 90.0
  * level                    (level) uint64 104B 50 100 150 200 ... 850 925 1000
  * lon                      (lon) float64 3kB 0.0 1.0 2.0 ... 357.0 358.0 359.0
    number                   int64 8B ...
  * time                     (time) timedelta64[ns] 224B 0 days 06:00:00 ... ...
Dimensions without coordinates: batch
Data variables:
    2m_temperature           (batch, time, lat, lon) float32 7MB ...
    mean_sea_level_pressure  (batch, time, lat, lon) float32 7MB ...
    10m_v_component_of_wind  (batch, time, lat, lon) float32 7MB ...
    10m_u_component_of_wind  (batch, time, lat, lon) float32 7MB ...
    total_precipitation_6hr  (batch, time, lat, lon) float32 7MB ...
    temperature              (batch, time, level, lat, lon) float32 95MB ...
    geopotential             (batch, time, level, lat, lon) float32 95MB ...
    u_component_of_wind      (batch, time, level, lat, lon) float32 95MB ...
    v_component_of_wind      (batch, time, level, lat, lon) float32 95MB ...
    vertical_velocity        (batch, time, level, lat, lon) float32 95MB ...
    specific_humidity        (batch, time, level, lat, lon) float32 95MB ...
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2025-08-05T06:24 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [5]:
for file in files[1:]:
    temp = xr.open_dataset(file)
    combined = xr.concat(combined, temp)
combined

/tmp/ipykernel_3573716/3071381339.py:2: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  temp = xr.open_dataset(file)


TypeError: can only concatenate xarray Dataset and DataArray objects, got <class 'str'>

In [2]:
base_targets0  = xr.open_dataset('/Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/target_init_2024-08-01 00:00:00.nc')
base_targets0

/tmp/ipykernel_3572205/649317930.py:1: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  base_targets0  = xr.open_dataset('/Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/target_init_2024-08-01 00:00:00.nc')


<xarray.Dataset> Size: 606MB
Dimensions:                  (batch: 1, time: 28, lat: 181, lon: 360, level: 13)
Coordinates:
    expver                   (time) <U4 448B ...
  * lat                      (lat) float64 1kB -90.0 -89.0 -88.0 ... 89.0 90.0
  * level                    (level) uint64 104B 50 100 150 200 ... 850 925 1000
  * lon                      (lon) float64 3kB 0.0 1.0 2.0 ... 357.0 358.0 359.0
    number                   int64 8B ...
  * time                     (time) timedelta64[ns] 224B 0 days 06:00:00 ... ...
Dimensions without coordinates: batch
Data variables:
    2m_temperature           (batch, time, lat, lon) float32 7MB ...
    mean_sea_level_pressure  (batch, time, lat, lon) float32 7MB ...
    10m_v_component_of_wind  (batch, time, lat, lon) float32 7MB ...
    10m_u_component_of_wind  (batch, time, lat, lon) float32 7MB ...
    total_precipitation_6hr  (batch, time, lat, lon) float32 7MB ...
    temperature              (batch, time, level, lat, lon) float32 95MB ...
    geopotential             (batch, time, level, lat, lon) float32 95MB ...
    u_component_of_wind      (batch, time, level, lat, lon) float32 95MB ...
    v_component_of_wind      (batch, time, level, lat, lon) float32 95MB ...
    vertical_velocity        (batch, time, level, lat, lon) float32 95MB ...
    specific_humidity        (batch, time, level, lat, lon) float32 95MB ...
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2025-08-05T06:24 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [16]:
base_targets0['total_precipitation_6hr'].sel(lat=slice(6, 38), lon=slice(65, 95)).isel(time=0).sum()

<xarray.DataArray 'total_precipitation_6hr' ()> Size: 4B
array(2.5391986, dtype=float32)
Coordinates:
    expver   <U4 16B ...
    number   int64 8B ...
    time     timedelta64[ns] 8B 06:00:00

In [26]:
base_targets0['total_precipitation_6hr'].sel(lat=slice(6, 38), lon=slice(65, 95)).sum(dim=['lat', 'lon']).values

array([[2.5391986, 2.5391986, 2.5391986, 2.1324644, 2.1324644, 2.1324644,
        2.1324644, 1.9960619, 1.9960619, 1.9960619, 1.9960619, 2.1042366,
        2.1042366, 2.1042366, 2.1042366, 2.0074825, 2.0074825, 2.0074825,
        2.0074825, 2.1016152, 2.1016152, 2.1016152, 2.1016152, 1.5071199,
        1.5071199, 1.5071199, 1.5071199, 1.079798 ]], dtype=float32)

In [20]:
skill_csv = pd.read_csv("skill_score_India_2025-08-1814-19-49.csv")
print(skill_csv)

                init_date             forecast_horizon  \
0     2014-08-01 00:00:00   21600000000000 nanoseconds   
1     2014-08-01 00:00:00   43200000000000 nanoseconds   
2     2014-08-01 00:00:00   64800000000000 nanoseconds   
3     2014-08-01 00:00:00   86400000000000 nanoseconds   
4     2014-08-01 00:00:00  108000000000000 nanoseconds   
...                   ...                          ...   
6043  2014-09-23 00:00:00  518400000000000 nanoseconds   
6044  2014-09-23 00:00:00  540000000000000 nanoseconds   
6045  2014-09-23 00:00:00  561600000000000 nanoseconds   
6046  2014-09-23 00:00:00  583200000000000 nanoseconds   
6047  2014-09-23 00:00:00  604800000000000 nanoseconds   

                                                  model region           mse  
0                  Graphcast_Base_2014-08-01_2014-09-30  India  4.794306e-10  
1                  Graphcast_Base_2014-08-01_2014-09-30  India  8.909178e-10  
2                  Graphcast_Base_2014-08-01_2014-09-30  India  8.

In [ ]:

all_results = []
initialization_dates = pd.to_datetime(pd.date_range(start=eval_start, end=eval_end, freq='D'))


logging.info(f"Starting evaluation for {len(initialization_dates)} initialization dates.")
all_results = []
idx = 0
for init_date in tqdm(initialization_dates, desc="Evaluating Forecasts"):
    logging.info(f"Processing initialization date: {init_date}")

    # 1. Load data for this initialization
    try:
        # Graphcast requires two time steps for input: the init time and 6 hours prior
        start_slice = init_date - pd.Timedelta(hours=6)
        end_slice = init_date + pd.Timedelta(days=7)

        start_slice -= select_time_eval.datetime.values[0][0]
        end_slice -= select_time_eval.datetime.values[0][0]

        # Use .copy(deep=True) to avoid memory issues with repeated slicing
        eval_sim_data = select_time_eval.sel(time=slice(start_slice, end_slice)).copy(deep=True)

        if eval_sim_data.sizes["time"] < 2:
            logging.warning(f"Not enough data for initialization {init_date}. Found {eval_sim_data.sizes['time']} steps. Skipping.")
            continue
    except Exception as e:
        logging.warning(f"Could not load data for {init_date}: {e}. Skipping.")
        continue


    from dask.diagnostics import ProgressBar
    with ProgressBar():
        eval_sim_data = select_time_eval.sel(time=slice(start_slice, end_slice)).compute()

    # 2. Prepare inputs, targets, and forcings for a 7-day rollout
    eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
        eval_sim_data, target_lead_times=target_lead_times_slice, **dataclasses.asdict(task_config))
    
    print("Eval Inputs:   ", eval_inputs.dims.mapping)
    print("Eval Targets:  ", eval_targets.dims.mapping)
    print("Eval Forcings: ", eval_forcings.dims.mapping)

    if (eval_inputs.dims.mapping['time'] != 2):
        print("\nError in extracting inputs targets and forcings\n")
        continue

    # Ensure we have enough target data for the longest forecast
    if eval_targets.sizes['time'] < MAX_FORECAST_STEPS:
        logging.warning(f"Not enough target data for a 7-day forecast from {init_date}. Have {eval_targets.sizes['time']} steps. Skipping.")
        continue

    targets_template = eval_targets * np.nan
    ground_truth_var = eval_targets[eval_vars]

    # 3. Run all Graphcast models
    logging.debug(f"Running Graphcast models for {init_date}")
    models_to_eval = {
    f"Graphcast_Base_{eval_start}_{eval_end}": run_model(params, state, eval_inputs, targets_template, eval_forcings)
}

    for i, (path, p) in enumerate(new_params_list, start=1):
        model_name = f"Graphcast_Finetuned_{i}_{os.path.basename(path)}"
        print(f"Running Model Name")
        models_to_eval[model_name] = run_model(p, state, eval_inputs, targets_template, eval_forcings)

    # print("Base run")
    # predictions_base = run_model(params, state, eval_inputs, targets_template, eval_forcings)
    # print("Finetuned 1")
    # predictions_ft1 = run_model(new_params1, state, eval_inputs, targets_template, eval_forcings)
    # print("Finetuned 2")
    # predictions_ft2 = run_model(new_params2, state, eval_inputs, targets_template, eval_forcings)
    
    # models_to_eval = {
    #     'Graphcast_Base' + eval_start + eval_end: predictions_base,
    #     'Graphcast_Finetuned1' + params_path_new1: predictions_ft1,
    #     'Graphcast_Finetuned2' + params_path_new2: predictions_ft2,
    # }

    # 4. Load, regrid, and align HRES forecast for the same initialization date
    hres_predictions = None
    hres_file_path = hres_files_map.get(init_date.to_pydatetime().replace(hour=0, minute=0, second=0, microsecond=0))
    if hres_file_path:
        try:
            logging.debug(f"Processing HRES file: {hres_file_path}")
            hres_ds = xr.open_dataset(hres_file_path, engine='cfgrib')
            hres_regridded = regrid_hres_fine_to_coarse(hres_ds, variable='tp', coarse_resolution=1.0)
            # Align time dimension with Graphcast targets
            hres_predictions = hres_regridded.drop_vars({'time'}).rename({'step': 'time'}).isel(time=slice(1,None)).assign_coords(time=eval_targets.time)
            models_to_eval['HRES'] = hres_predictions
        except Exception as e:
            logging.warning(f"Failed to process HRES file {hres_file_path}: {e}")
    else:
        logging.warning(f"No HRES file found for init date {init_date}")

        

    for model_name, predictions in tqdm(models_to_eval.items(), desc="MSE & Diff Calc"):
        if eval_vars in predictions:
            pred_var = predictions[eval_vars]
        else:
            pred_var = predictions

        times = pred_var.time.values

        # *** CHANGE: loop over each region
        for region in tqdm(world_regions, desc=f"World regions for {model_name}"):
            for timestep in times:
                # Slice predictions and targets to the current forecast horizon
                pred_sliced = pred_var.sel(time=timestep)
                targ_sliced = ground_truth_var.sel(time=timestep)

                # Compute difference for this region
                diff = pred_sliced - targ_sliced
                # *** CHANGE: apply region mask per region name
                diff_region = mask_dbase_india_buffer(diff)

                # Compute MSE over the region
                mse = float((diff_region**2).mean())
                rmse = float(np.sqrt(mse))

                # Store MSE result
                result_row = {
                    'init_date': init_date.strftime('%Y-%m-%d %H:%M:%S'),
                    'forecast_horizon': str(timestep),
                    'model': model_name,
                    'region': region,                         # *** CHANGE: include region
                    'rmse': rmse
                }
                all_results.append(result_row)

                # *** CHANGE: write MSE CSV per region
                csv_filename = f'skill_score_{region}_{current_date}.csv'
                pd.DataFrame([result_row]).to_csv(
                    csv_filename,
                    mode='a',
                    header=not os.path.exists(csv_filename),
                    index=False
                )

                logging.debug(f"[{region}] {init_date}, {model_name}, {timestep} RMSE = {rmse:.6f}")

                # *** CHANGE: also save the full diff xarray for later inspection
                diff_ds = diff_region.to_dataset(name='difference')
                # annotate coords or attrs so you know init/model/region in the file
                diff_ds.attrs.update({
                    'init_date': init_date.strftime('%Y-%m-%d %H:%M:%S'),
                    'model': model_name,
                    'region': region,
                    'forecast_horizon': str(timestep)
                })
                nc_filename = (
                    f'diff_{region}_{model_name}_'
                    f"{init_date.strftime('%Y%m%d%H')}_t{timestep}.nc"
                )
                print("Diff filesize:")
                print(diff_ds.nbytes)
                diff_ds.to_netcdf(nc_filename)
                logging.debug(f"Saved diff xarray to {nc_filename}")

# 6. Save all results to a CSV file
logging.info("Evaluation loop finished. Saving results to CSV.")
results_df = pd.DataFrame(all_results)

output_dir = os.path.dirname(output_csv_path)
if output_dir:
    os.makedirs(output_dir, exist_ok=True)

results_df.to_csv(output_csv_path, index=False)
logging.info(f"Evaluation complete. Results saved to skill_score.csv and args csv")
print("\n--- Sample of Evaluation Results ---")
print(results_df.head())
print("------------------------------------\n")


# =============================================================================
# === OLD PLOTTING-FOCUSED LOOP (COMMENTED OUT) ===============================
# =============================================================================
"""
# Helper functions for plotting (can be moved to a utils file)
def accumulate_diff(diff_list, use_abs=False):
    # ... (function definition from original script)
def plot_accumulated_diff(data_array, title, simnum, ...):
    # ... (function definition from original script)
def plot_accumulated_diff_side_by_side(truth, pred1, pred2, ...):
    # ... (function definition from original script)
    
logging.info("Starting original plotting loop for a few samples...")
for i, hres_file in tqdm(enumerate(sorted_paths[5:11]), desc="Sims"):
    # This block is now superseded by the systematic evaluation loop above.
    # It can be used for debugging or generating sample plots for a few specific dates.
    # ... (original loop code) ...
"""

# </eval_forecast_save.py>